In [1]:
"""
Test script for LMaaS
"""
from openai import AzureOpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam

import config
from idam_token_generator import IDAMTokenGenerator


idam = IDAMTokenGenerator(
    config.IDAM_TOKEN_ENDPOINT,
    config.IDAM_APP_CLIENT_ID,
    config.IDAM_APP_CLIENT_SECRET,
    config.IDAM_LMAAS_APP_AUDIENCE
)


llm = AzureOpenAI(
        azure_endpoint = config.OPENAI_ENDPOINT,
        azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
        api_version = config.OPENAI_AZURE_API_VERSION,
        azure_ad_token = idam.get_idam_token()
    )


Expiry_time: %s 1761418479
exp_time : %s 2025-10-25 18:54:39+00:00
current_time : %s 2025-10-25 18:49:53.480592+00:00
JWT token is VALID
Reusing existing valid token.


In [2]:
messages = [
    ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant."),
    ChatCompletionUserMessageParam(role="user", content="What is the capital of France?"),
]

response = llm.chat.completions.create(
        model = config.OPENAI_DEPLOYMENT_MODEL,
        messages = messages,
    )

print(response.choices[0].message.content)

Paris.


In [3]:
# load the annotations data
import json
annotation_path01 = "/qumulo/shared_data/aofei_summer/RegTok/data/BiomedParse_SegVQA.json"

annotations = []
# load jsonlines
with open(annotation_path01, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

In [4]:
len(annotations), list(annotations.keys())[0]

(1970,
 '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_5924')

In [5]:
annotations['/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_5924']

{'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0325_147_CT_liver.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_mask/amos_0325_147_CT_liver_liver.png',
   'area': 48760,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_5924',
   'bbox': [152, 384, 281, 309],
   'category_id': 1,
   'id': 17853,
   'slice_ratio': 0.4,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0325_147_CT_liver.png',
   'split': 'test',
   'sentences': [{'raw': 'liver in liver CT',
     'sent': 'liver in liver CT',
     'sent_id': 35343}],
   'sent_ids': [35343],
   'ann_id': 17853,
   'ref_id': 17853,
   'modality_label': 0}]}

In [6]:
image_H, image_W = 1024, 1024
def preprocess_annotations(annotation):
    processed = []
    processed_with_info = []
    for region in annotation['mask_annotations']:
        item = {
            "id": region['id'],
        }
        item_with_info = {
            "id": region['id'],
            "mask_file": region.get("mask_file", ""),
            "image_id": region.get("image_id", "")
        }

        processed_bbox = [
            round(region['bbox'][1] / image_W, 3),
            round(region['bbox'][0] / image_H, 3),
            round(region['bbox'][3] / image_W, 3),
            round(region['bbox'][2] / image_H, 3)
        ]
        # item['bbox'] = processed_bbox
        processed_sentences = []
        for sentence in region['sentences']:
            processed_sentences.append(sentence['raw'])
        item['sentences'] = processed_sentences
        processed.append(item)
        processed_with_info.append(item_with_info)
    return processed, processed_with_info

In [7]:
sampled_image_id = 0
processed_items = []
processed_items_with_info = []
for k in annotations:
    processed, processed_with_info = preprocess_annotations(annotation=annotations[k])
    annotation = annotations[k]
    image_masks = {
        "image_id": sampled_image_id,
        "masks": processed
    }
    processed_items.append(image_masks) 
    image_masks_info = {
        "image_id": sampled_image_id,
        "image_file": annotation.get("image_file", ""),
        "num_masks": len(processed),
        "mask_id": [(item['id'], item['mask_file']) for item in processed_with_info]
    }

    processed_items_with_info.append(image_masks_info)
    sampled_image_id += 1

In [8]:
processed_items[1], processed_items_with_info[1]

({'image_id': 1,
  'masks': [{'id': 10976, 'sentences': ['bladder in abdominal CT']}]},
 {'image_id': 1,
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0292_36_CT_abdomen.png',
  'num_masks': 1,
  'mask_id': [(10976,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_mask/amos_0292_36_CT_abdomen_bladder.png')]})

In [ ]:
Alignment_prompt = """
You are an expert radiologist and dataset curator. 
You are given region-level annotations for one or more medical images. Each image item includes:
- "image_id": an integer id,
- "masks": a list of region objects, each with:
    - "id": id of this mask
    - "sentences": free-text descriptions (list of strings).

Assume you can see the image implicitly and must use only the provided annotation information (sentences) to perform the tasks below. 

Task (produce a single JSON output per image):
- Generate evaluation data in the form of VQA about the image and segmentation based on the provided region information.
Follow the style of radiology and clinical reasoning, and make sure the generated questions are natural, factual, and unambiguous.

Output rules and format (strict):
- Always return a JSON list containing one object per input image: [{...}, {...}, ...].
- Each image object must contain:
  {
    "image_id": <the input image_id>,
    "QAs": [ <list of Q&A objects> ]
  }
- Each dialogue turn is a JSON object:
  {
    "User": "<user question>",
    "Assistant": "<assistant answer>",
    "mask_ids": [ <list of mask indices appearing in the answer> ],
    "Question_type": "open" or "close"
  }

QA generation rules:
- For each image, generate one Q&A pair for each provided mask.
- Each QA serves with 2 evaluation targets: VQA and segmentation.
- For each QA, include the answer types (open or close), where open means open-ended, and close means "yes"/"no".
- For each QA, include a short version of answer for the convenience of evaluation VQA (e.g., yes/no for close-ended, short diagnosis for open-ended).
- For the mask with only organ, you may use close-ended questions to ask the existence.
- For the mask with abnormalities, you may use open-ended questions to ask the diagnosis.
- You do not have to enumerate all the mentioned masks or organs, focus on main organs, findings and control the totoal number of QAs for each image less than 5.

Additional rules for diverse QA generation:
- If the image only contains normal organs (no abnormalities mentioned in any mask sentences):
  - Generate about 50% questions that have the answer "No" to balance the positive and negative labels.
- For normal organs with no abnormality, you can choose to generate questions that explicitly request segmentation before the diagnostic check.
- Example question pattern:
  - "Please segment and focus on the liver, then determine if there is any lesion or abnormality."
  - "First segment the pancreas, then state whether it appears normal or abnormal."
- Example answers:
  - "The liver is segmented as <mask>. It appears normal with no lesion." (short answer: "No")
  - "The pancreas is segmented as <mask>. No mass or cyst is observed." (short answer: "No")
- Always include the corresponding mask_id(s) in the output, even when the finding is normal.

Example input of one image (for reference only):
{
  "image_id": 0,
  "masks": [
    {"id": 0, "sentences": ["liver"]},
    {"id": 1, "sentences": ["tumor"]},
    {"id": 2, "sentences": ["spleen"]},
  ]
}

Example output (for one image):
[
  {
    "image_id": 0,
    "QAs": [
      {
        "User": "Is there a liver in this CT scan? If so, please segment it.",
        "Assistant": "Yes. The liver is segmented as <mask>.",
        "short answer": "Yes",
        "mask_ids": [0],
        "Question_type": "close"
      },
      {
        "User": "What abnormality is seen on the liver? Please do diagnosis and then segment it if it exists.",
        "Assistant": "There is a tumor at the center right lobe, segmented as <mask> inside the liver.",
        "mask_ids": [1],
        "short answer": "Tumor",
        "Question_type": "open"
      },
      {
        "User": "Please segment and focus on the spleen, then determine if there is any lesion or abnormality.",
        "Assistant": "The spleen is segmented as <mask> and there is no lesion or abnormality observed.",
        "short answer": "No",
        "mask_ids": [2],
        "Question_type": "close"
      },
    ]
  }
]

Please do not mention words like "annotations", "annotated regions" in the generated QA.

Finally: The API will provide the "masks" list as input. Produce the JSON outputs (one object per image) strictly following the rules above. Do not include any extra text outside the JSON list in the model's final reply. """


In [10]:
list_outputs = []
# list_outputs = list_outputs[:100]

In [11]:
output_json_file = "BiomedParse_SegVQA_GPTV2.jsonl"
ans_file = open(output_json_file, "a")

In [12]:
len(processed_items)

1970

In [13]:
batch_size = 10
max_retry = 3
from tqdm import tqdm
# for i in tqdm(range(0, len(processed_items), batch_size)):
for i in tqdm(range(0, 10, batch_size)):
    num_try = 1
    items = processed_items[i:i + batch_size]
    before_process_items = processed_items_with_info[i:i + batch_size]
    messages = [
        ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant." + Alignment_prompt + "\n\n"),
        ChatCompletionUserMessageParam(role="user", content="The input with multiple images:" + str(items)),
    ]
    try:
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
    except:
        idam = IDAMTokenGenerator(
            config.IDAM_TOKEN_ENDPOINT,
            config.IDAM_APP_CLIENT_ID,
            config.IDAM_APP_CLIENT_SECRET,
            config.IDAM_LMAAS_APP_AUDIENCE
        )


        llm = AzureOpenAI(
                azure_endpoint = config.OPENAI_ENDPOINT,
                azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
                api_version = config.OPENAI_AZURE_API_VERSION,
                azure_ad_token = idam.get_idam_token()
            )
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
        num_try += 1
        if num_try > max_retry:
            print(f"Max retries exceeded for batch starting at index {i}")
            continue

    # print(response.choices[0].message.content)
    llm_out = response.choices[0].message.content
    llm_out_json = json.loads(llm_out)
    items = processed_items[i:i + batch_size]
    original_items = processed_items_with_info[i:i + batch_size]
    for j in range(batch_size):
        original_item = original_items[j]
        llm_out_json[j]['image_file'] = original_item['image_file']
        llm_out_json[j]['mask_id'] = original_item['mask_id']
    list_outputs.extend(llm_out_json)
    ans_file.write("\n".join([json.dumps(x) for x in llm_out_json]) + "\n")
    ans_file.flush()
# ans_file.close()


  0%|          | 0/1 [00:00<?, ?it/s]

Expiry_time: %s 1761418479
exp_time : %s 2025-10-25 18:54:39+00:00
current_time : %s 2025-10-25 18:55:04.664619+00:00
JWT token is NOT VALID
Generating new token.


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Access Token is generated


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Exchange Access Token is generated


100%|██████████| 1/1 [01:00<00:00, 60.24s/it]


In [14]:
llm_out_json

[{'image_id': 0,
  'QAs': [{'User': 'Please segment and focus on the liver in this liver CT, then determine if there is any focal lesion or abnormality.',
    'Assistant': 'The liver is segmented as <mask>. No focal lesion or abnormality is detected.',
    'short answer': 'No',
    'mask_ids': [17853],
    'Question_type': 'close'}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0325_147_CT_liver.png',
  'mask_id': [(17853,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_mask/amos_0325_147_CT_liver_liver.png')]},
 {'image_id': 1,
  'QAs': [{'User': 'Is the urinary bladder present on this abdominal CT? If so, please segment it.',
    'Assistant': 'Yes. The urinary bladder is segmented as <mask>.',
    'short answer': 'Yes',
    'mask_ids': [10976],
    'Question_type': 'close'}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0292_36_CT_abdomen.png',
  'mask_id': [